# Using DuckDB for Data Analysis

## Overview

In this exercise, you will walk through the previous data preprocessing workflow using DuckDB.

The goal is to replicate the same steps we performed in polars, but this time using SQL queries in DuckDB. This will give you a chance to see how the same transformations can be expressed in SQL and how DuckDB handles data manipulation.

You will learn how to:

- Query a Parquet file with SQL and inspect its schema
- Check for missing values with `COUNT`
- Convert a text label to a boolean
- Build flag columns, per-user aggregates, and window features
- Test a simple fraud rule against the fraud label, and check that DuckDB and Polars flag exactly the same rows

## Import Libraries


In [1]:
import duckdb

## Read the Input Parquet File

First, download the `credit_card_transactions-ibm_v2.parquet` file from [the course datasets repository](https://media.githubusercontent.com/media/bdi593/datasets/main/credit-card-transactions-ibm/credit_card_transactions-ibm_v2.parquet) and place it in the same folder as your Jupyter notebook. If you ran the Polars notebook, it has already written this file for you.

The file is about 250 MB, which is about 9 times smaller than the original CSV file, making it much faster to read and process.

:::{tip} Why is Parquet smaller than CSV?

Recall that Parquet is a columnar storage format that uses efficient compression and encoding techniques, while CSV is a plain text format that stores data in a row-wise manner without any compression. This is why Parquet files are often significantly smaller than their CSV counterparts.

You can also compress CSV files using different algorithms (like gzip or bzip2), but even then, Parquet often achieves better compression ratios due to its columnar nature and optimized encoding.

:::

### `duckdb.connect()`

The `duckdb.connect()` function creates a connection to a DuckDB database. If you don't specify a database name, it creates an in-memory database that exists only for the duration of the connection. This is useful for ad-hoc analysis and temporary data manipulation without needing to manage a persistent database file.

We also turn off DuckDB's progress bar. When the `ipywidgets` package is installed, DuckDB shows the progress of long queries as an interactive widget, which does not display on a static web page such as this textbook. Without `ipywidgets` the progress bar is already off, and DuckDB refuses to change the setting, so the code ignores that error.

In [2]:
con = duckdb.connect()

# Hide the progress bar widget so long queries print only their results.
try:
    con.execute("SET enable_progress_bar = false")
except duckdb.InvalidInputException:
    pass  # ipywidgets is not installed, so the progress bar is already off.

con.execute(
    """
CREATE OR REPLACE VIEW transactions AS
SELECT *
FROM read_parquet('credit_card_transactions-ibm_v2.parquet');
"""
)

# Peek (like df.head())
con.sql("SELECT * FROM transactions LIMIT 5").df()

,User,Card,Amount,Datetime,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,134.09,2002-09-01 06:21:00,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,None,No
1,0,0,38.48,2002-09-01 06:42:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No
2,0,0,120.34,2002-09-02 06:22:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No
3,0,0,128.95,2002-09-02 17:45:00,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,None,No
4,0,0,104.71,2002-09-03 06:23:00,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,None,No


:::{tip} DuckDB's `.df()` Method

The `.df()` method in DuckDB is used to convert a DuckDB query result into a Pandas DataFrame. This allows you to easily manipulate and analyze the data using Pandas' powerful data handling capabilities after performing SQL queries in DuckDB.

However, you should be cautious when using `.df()` with large datasets, as it will load the entire result set into memory. If the dataset is too large to fit in memory, it can lead to performance issues or crashes. Always consider the size of your data and the available memory before using `.df()`.

To avoid loading large dataset into memory, consider (1) using the `LIMIT` clause in your SQL queries to reduce the amount of data being loaded when testing or exploring the data, and (2) aggregating or filtering the data in SQL to reduce the result set size before converting it to a DataFrame.

:::


## Exploratory Data Analysis (EDA)


### Describe Schema

To understand the structure of the dataset, we can use the `DESCRIBE` command to get information about the columns, their data types, and whether they contain null values.


In [3]:
con.sql("DESCRIBE transactions;").df()

,column_name,column_type,null,key,default,extra
0,User,BIGINT,YES,None,None,None
1,Card,BIGINT,YES,None,None,None
2,Amount,DOUBLE,YES,None,None,None
3,Datetime,TIMESTAMP,YES,None,None,None
4,Use Chip,VARCHAR,YES,None,None,None
5,Merchant Name,BIGINT,YES,None,None,None
6,Merchant City,VARCHAR,YES,None,None,None
7,Merchant State,VARCHAR,YES,None,None,None
8,Zip,DOUBLE,YES,None,None,None
9,MCC,BIGINT,YES,None,None,None


If you're familiar with the `PRAGMA` introspection command from SQLite or another engine, you can also use `PRAGMA table_info('transactions');` to get similar information about the columns in the `transactions` table.


In [4]:
con.sql("PRAGMA table_info('transactions');").df()

,cid,name,type,notnull,dflt_value,pk
0,0,User,BIGINT,False,None,False
1,1,Card,BIGINT,False,None,False
2,2,Amount,DOUBLE,False,None,False
3,3,Datetime,TIMESTAMP,False,None,False
4,4,Use Chip,VARCHAR,False,None,False
5,5,Merchant Name,BIGINT,False,None,False
6,6,Merchant City,VARCHAR,False,None,False
7,7,Merchant State,VARCHAR,False,None,False
8,8,Zip,DOUBLE,False,None,False
9,9,MCC,BIGINT,False,None,False


Note that the `Datetime` column is `TIMESTAMP` type, which means it contains both date and time information. The `Amount` column is `DOUBLE`, which is a floating-point number, and the `Merchant Name` column is `BIGINT`, which were encoded as integers to avoid confusion with real merchant names. The `Errors?` and `Is Fraud?` columns use `VARCHAR` type, which is a variable-length string.


In [5]:
con.sql('SELECT DISTINCT("Is Fraud?") FROM transactions;').df()

,Is Fraud?
0,No
1,Yes


### Check for missing values

`COUNT(*)` counts every row, while `COUNT(column)` skips nulls. The difference between the two is the number of missing values in that column. The Polars notebook shows that only these three columns have missing values.

In [6]:
con.sql(
    """
SELECT
  COUNT(*) - COUNT("Merchant State") AS missing_merchant_state,
  COUNT(*) - COUNT("Zip")            AS missing_zip,
  COUNT(*) - COUNT("Errors?")        AS missing_errors
FROM transactions;
"""
).df()

,missing_merchant_state,missing_zip,missing_errors
0,2720821,2878135,23998469


As in the Polars notebook, ask _why_ a value is missing before you drop or fill it. The query below groups the transactions with no `"Zip"` by the type of merchant.

In [7]:
con.sql(
    """
SELECT
  CASE
    WHEN "Merchant City" = 'ONLINE' THEN 'Online merchant'
    ELSE 'Merchant outside the U.S.'
  END      AS merchant_type,
  COUNT(*) AS n_missing_zip
FROM transactions
WHERE "Zip" IS NULL
GROUP BY merchant_type
ORDER BY n_missing_zip DESC;
"""
).df()

,merchant_type,n_missing_zip
0,Online merchant,2720821
1,Merchant outside the U.S.,157314


The counts match the Polars notebook. `"Merchant State"` and `"Zip"` are empty for online merchants, and `"Zip"` is also empty for merchants outside the U.S. `"Errors?"` is empty when the transaction had no error. These nulls carry meaning, so we leave them as they are.

### Convert `"Is Fraud?"` column to a boolean

The `"Is Fraud?"` column currently contains string values ("Yes" and "No"). We can convert this column to a boolean type for easier analysis.


In [8]:
con.execute(
    """
CREATE OR REPLACE VIEW transactions_with_fraud_flag AS
SELECT
  * EXCLUDE ("Is Fraud?"),
  CASE
    WHEN "Is Fraud?" = 'Yes' THEN TRUE
    WHEN "Is Fraud?" = 'No'  THEN FALSE
    ELSE NULL
  END AS "Is Fraud?"
FROM transactions;
"""
)

con.sql(
    """
SELECT *
FROM transactions_with_fraud_flag
LIMIT 5;
"""
).df()

,User,Card,Amount,Datetime,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,134.09,2002-09-01 06:21:00,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,None,False
1,0,0,38.48,2002-09-01 06:42:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,False
2,0,0,120.34,2002-09-02 06:22:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,False
3,0,0,128.95,2002-09-02 17:45:00,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,None,False
4,0,0,104.71,2002-09-03 06:23:00,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,None,False


You can verify that the `"Is Fraud?"` column is now a boolean type using the `DESCRIBE` command again.


In [9]:
con.sql("DESCRIBE transactions_with_fraud_flag;").df()

,column_name,column_type,null,key,default,extra
0,User,BIGINT,YES,None,None,None
1,Card,BIGINT,YES,None,None,None
2,Amount,DOUBLE,YES,None,None,None
3,Datetime,TIMESTAMP,YES,None,None,None
4,Use Chip,VARCHAR,YES,None,None,None
5,Merchant Name,BIGINT,YES,None,None,None
6,Merchant City,VARCHAR,YES,None,None,None
7,Merchant State,VARCHAR,YES,None,None,None
8,Zip,DOUBLE,YES,None,None,None
9,MCC,BIGINT,YES,None,None,None


### Payment method frequency analysis

The code below provides a high-level summary of the payment methods used across your credit card dataset. The `GROUP BY` query counts the rows for each unique value of the `"Use Chip"` column (Chip, Online, and Swipe transactions). It is the SQL counterpart of Polars' `.value_counts()`.


In [10]:
con.sql(
    """
SELECT
  "Use Chip",
  COUNT(*) AS count
FROM transactions
GROUP BY "Use Chip"
ORDER BY count DESC;
        """
)

Use ChipVARCHAR,countBIGINT
Swipe Transaction,15386082
Chip Transaction,6287598
Online Transaction,2713220


Looking at the numbers, you can see that Swipe Transactions are the dominant payment method in this dataset with over 15 million entries, followed by Chip Transactions at roughly 6.3 million and Online Transactions at about 2.7 million. This breakdown is essential for understanding consumer behavior or detecting potential fraud patterns, as certain types of transactions (like "Online" vs. "Swipe") carry different risk profiles.


### Fraud distribution analysis

The code below shows the frequency of the `"Is Fraud?"` column to determine the distribution of legitimate versus fraudulent transactions within your dataset.


In [11]:
con.sql(
    """
SELECT
  "Is Fraud?",
  COUNT(*) AS count
FROM transactions
GROUP BY "Is Fraud?"
ORDER BY count DESC;
        """
)

Is Fraud?VARCHAR,countBIGINT
No,24357143
Yes,29757


The data reveals that the overwhelming majority of transactions - over 24.3 million - are flagged as "No", while only 29,757 are flagged as "Yes". While the fraudulent cases represent a very small fraction of the total volume (roughly 0.12%), identifying this minority is the primary goal of most credit card analytics. Understanding this ratio is a critical first step before building a machine learning model, as it tells you that you'll need specialized techniques, like oversampling or specific loss functions, to account for the rarity of fraud.


### Flag suspicious transactions based on amount

The code below performs feature engineering by creating three new boolean (`True`/`False`) indicators based on specific transaction characteristics.

The `AS` keyword is used to assign a name to each new column, making it easier to reference them in subsequent queries.


In [12]:
con.sql(
    """
SELECT
  *,
  "Amount" < 0 AS is_refund,
  "Amount" > 500 AS large_txn,
  "Use Chip" = 'Online Transaction' AS is_online
FROM transactions
LIMIT 10;
"""
).df()

,User,Card,Amount,Datetime,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?,is_refund,large_txn,is_online
0,0,0,134.09,2002-09-01 06:21:00,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,None,No,False,False,False
1,0,0,38.48,2002-09-01 06:42:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No,False,False,False
2,0,0,120.34,2002-09-02 06:22:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No,False,False,False
3,0,0,128.95,2002-09-02 17:45:00,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,None,No,False,False,False
4,0,0,104.71,2002-09-03 06:23:00,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,None,No,False,False,False
5,0,0,86.19,2002-09-03 13:53:00,Swipe Transaction,-7146670748125200898,Monterey Park,CA,91755.0,5970,None,No,False,False,False
6,0,0,93.84,2002-09-04 05:51:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No,False,False,False
7,0,0,123.50,2002-09-04 06:09:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No,False,False,False
8,0,0,61.72,2002-09-05 06:14:00,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,None,No,False,False,False
9,0,0,57.10,2002-09-05 09:35:00,Swipe Transaction,4055257078481058705,La Verne,CA,91750.0,7538,None,No,False,False,False


These "flag" columns can be useful for both exploratory analysis and machine learning. For example, you can check whether "large transactions" are more likely to be "fraudulent", or compare the total volume of "online" versus in-person sales. This step transforms raw data into behavioral features, providing the model or the analyst with clear, binary signals that highlight high-interest events within the transaction stream.

Note that in SQL these flags exist only in the result of this one query. To reuse them, put the query in a view (`CREATE VIEW ... AS SELECT ...`) or a common table expression (`WITH ...`), as we do for the composite rule below.

### Filter rows based on conditions

You can also apply a filter to isolate a very specific subset of your data: online refunds. The `WHERE` clause in the SQL query allows you to specify multiple conditions that must be met for a row to be included in the result set. In this case, we are filtering for transactions where the `"Amount"` is less than 0 (indicating a refund) and the `"Use Chip"` column indicates an "Online Transaction".


In [13]:
con.sql(
    """
SELECT *
FROM transactions
WHERE
  "Amount" < 0
  AND "Use Chip" = 'Online Transaction'
LIMIT 10;
"""
).df()

,User,Card,Amount,Datetime,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,-100.0,2004-12-10 20:45:00,Online Transaction,7501849281341469857,ONLINE,None,NaN,4722,None,No
1,0,0,-393.0,2010-10-08 18:00:00,Online Transaction,333722291367506728,ONLINE,None,NaN,4722,None,No
2,0,0,-443.0,2011-12-19 12:50:00,Online Transaction,333722291367506728,ONLINE,None,NaN,4722,None,No
3,0,0,-473.0,2015-11-20 07:42:00,Online Transaction,-8566951830324093739,ONLINE,None,NaN,3640,None,Yes
4,0,2,-461.0,2017-02-21 08:03:00,Online Transaction,7501849281341469857,ONLINE,None,NaN,4722,None,No
5,0,3,-140.0,2003-02-27 23:24:00,Online Transaction,-6796080605569913348,ONLINE,None,NaN,4722,None,No
6,0,3,-246.0,2005-01-22 12:59:00,Online Transaction,3694722044710185708,ONLINE,None,NaN,4722,None,No
7,0,3,-450.0,2005-06-30 06:36:00,Online Transaction,-6796080605569913348,ONLINE,None,NaN,4722,None,No
8,0,3,-351.0,2008-03-09 06:32:00,Online Transaction,7501849281341469857,ONLINE,None,NaN,4722,None,No
9,0,3,-386.0,2013-01-13 21:45:00,Online Transaction,7501849281341469857,ONLINE,None,NaN,4722,None,No


### Per-user aggregation

The code below performs a user-level aggregation, collapsing millions of individual transaction records into a concise summary of spending behavior for each unique customer. By using the `GROUP BY` clause, you are instructing DuckDB to organize the data into buckets based on the individual user ID. The `SELECT` statement with aggregate functions then calculates three key metrics for each bucket:

1. the total volume of money spent,
2. the average cost per purchase, and
3. the total number of transactions


In [14]:
user_stats = con.sql(
    """
SELECT
  "User",
  SUM("Amount") AS total_spent,
  AVG("Amount") AS average_amount,
  COUNT(*)      AS num_transactions
FROM transactions
GROUP BY "User";
"""
).df()

user_stats.head()

,User,total_spent,average_amount,num_transactions
0,0,1622991.69,81.299989,19963
1,1,723491.89,81.118050,8919
2,2,1475933.36,35.159687,41978
3,3,1186497.51,117.277603,10117
4,4,1798790.91,97.011698,18542


Rolling / window features (SQL-like mental model):


### 10 most recent transactions rolling sum

You can also compute rolling aggregates to capture recent behavior. The code below calculates a rolling sum of the `"Amount"` column over the last 10 transactions for each user.

Window functions depend on row order, and `"Datetime"` has minute resolution: a user can make several transactions in the same minute. With `ORDER BY "Datetime"` alone, tied rows can be processed in any order, so results can change between runs and differ from Polars. We break ties the same way as the Polars notebook: by `"Card"`, then by the row's position in the source file. `read_parquet(..., file_row_number = true)` adds that position as a `file_row_number` column. We save it in a second view, `transactions_numbered`.

In [15]:
con.execute(
    """
CREATE OR REPLACE VIEW transactions_numbered AS
SELECT *
FROM read_parquet('credit_card_transactions-ibm_v2.parquet', file_row_number = true);
"""
)

con.sql(
    """
SELECT
  "User",
  "Card",
  "Datetime",
  "Amount",
  SUM("Amount") OVER (
    PARTITION BY "User"
    ORDER BY "Datetime", "Card", file_row_number
    ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
  ) AS rolling_amount_sum_10
FROM transactions_numbered
ORDER BY "User", "Datetime", "Card", file_row_number
LIMIT 12;
"""
).df()

,User,Card,Datetime,Amount,rolling_amount_sum_10
0,0,0,2002-09-01 06:21:00,134.09,134.09
1,0,0,2002-09-01 06:42:00,38.48,172.57
2,0,0,2002-09-02 06:22:00,120.34,292.91
3,0,0,2002-09-02 17:45:00,128.95,421.86
4,0,0,2002-09-03 06:23:00,104.71,526.57
5,0,0,2002-09-03 13:53:00,86.19,612.76
6,0,0,2002-09-04 05:51:00,93.84,706.60
7,0,0,2002-09-04 06:09:00,123.50,830.10
8,0,0,2002-09-05 06:14:00,61.72,891.82
9,0,0,2002-09-05 09:35:00,57.10,948.92


Compare the first rows with the Polars output. Polars' `rolling_sum(window_size=10)` returns `null` until a user has 10 transactions. The SQL frame `ROWS BETWEEN 9 PRECEDING AND CURRENT ROW` sums whatever rows are available, so the first nine rows hold partial sums. From the tenth transaction on, the two agree.

## Additional Analyses

The same business questions we explored in the Polars notebook can be
expressed concisely in SQL. The queries below extend the preprocessing
pipeline with a few more analyses you would routinely build for a fraud or
marketing team.


### Top Merchants by Total Spend

A textbook `GROUP BY ... ORDER BY` query.


In [16]:
con.sql(
    """
SELECT
    "Merchant Name",
    ROUND(SUM("Amount"), 2)  AS total_spend,
    COUNT(*)                 AS n_transactions,
    ROUND(AVG("Amount"), 2)  AS avg_ticket
FROM transactions
GROUP BY "Merchant Name"
ORDER BY total_spend DESC
LIMIT 10
"""
)

Merchant NameBIGINT,total_spendDOUBLE,n_transactionsBIGINT,avg_ticketDOUBLE
-4282466774399734331,101787066.82,1129061,90.15
1913477460590765860,35349420.24,562158,62.88
-2088492411650162548,25554655.3,720615,35.46
2027553650310142703,22045032.2,1028485,21.43
4241336128694185533,19182882.88,173206,110.75
-5467922351692495955,17096508.71,374905,45.6
1799189980464955940,16417797.54,1130230,14.53
-5162038175624867091,14271024.91,676694,21.09
97032797689821735,14245596.06,541794,26.29
4722913068560264812,12182126.2,476913,25.54


### Hourly Transaction Pattern

DuckDB's `EXTRACT(HOUR FROM ...)` returns the hour of day from a timestamp.
This is the SQL counterpart of the Polars `.dt.hour()` expression.


In [17]:
con.sql(
    """
SELECT
    EXTRACT(HOUR FROM "Datetime") AS hour,
    COUNT(*)                      AS n_transactions,
    ROUND(SUM("Amount"), 2)       AS total_spend
FROM transactions
GROUP BY hour
ORDER BY hour
"""
)

hourBIGINT,n_transactionsBIGINT,total_spendDOUBLE
0,235608,12522029.46
1,213411,11972821.86
2,214075,10928316.34
3,183370,9692236.4
4,203550,11400974.34
5,321588,14832093.09
6,1347058,47235870.39
7,1644618,55614757.51
8,1616851,57074988.17
9,1598928,56437859.53


### Fraud Rate by Use-Chip Category

This query uses the `transactions_with_fraud_flag` view from earlier, where `"Is Fraud?"` is a boolean. Casting a boolean to an integer turns `TRUE` into 1 and `FALSE` into 0, so `SUM` counts the fraud cases and `AVG` gives the share of rows that are fraud. This is equivalent to the Polars `pl.col("is_fraud").mean()` expression. On the original `'Yes'`/`'No'` text column, you would write `AVG(CASE WHEN "Is Fraud?" = 'Yes' THEN 1 ELSE 0 END)` instead.

In [18]:
con.sql(
    """
SELECT
    "Use Chip",
    COUNT(*)                                       AS n_transactions,
    SUM("Is Fraud?"::INTEGER)                      AS n_fraud,
    ROUND(AVG("Is Fraud?"::INTEGER) * 100, 3)      AS fraud_rate_pct
FROM transactions_with_fraud_flag
GROUP BY "Use Chip"
ORDER BY fraud_rate_pct DESC
"""
)

Use ChipVARCHAR,n_transactionsBIGINT,n_fraudHUGEINT,fraud_rate_pctDOUBLE
Online Transaction,2713220,18349,0.676
Chip Transaction,6287598,4836,0.077
Swipe Transaction,15386082,6572,0.043


### Time Since Previous Transaction (per User)

`LAG()` is the canonical window function for "give me the previous row's
value". Pair it with `PARTITION BY User ORDER BY Datetime, Card, file_row_number`
and you get exactly the same per-user shift we computed in Polars.

In [19]:
con.sql(
    """
SELECT
    "User",
    "Datetime",
    "Amount",
    EXTRACT(EPOCH FROM
        "Datetime" - LAG("Datetime") OVER (
            PARTITION BY "User"
            ORDER BY "Datetime", "Card", file_row_number
        )
    ) AS seconds_since_last_txn
FROM transactions_numbered
ORDER BY "User", "Datetime", "Card", file_row_number
LIMIT 15
"""
)

UserBIGINT,DatetimeTIMESTAMP,AmountDOUBLE,seconds_since_last_txnDOUBLE
0,2002-09-01 06:21:00,134.09,NULL
0,2002-09-01 06:42:00,38.48,1260.0
0,2002-09-02 06:22:00,120.34,85200.0
0,2002-09-02 17:45:00,128.95,40980.0
0,2002-09-03 06:23:00,104.71,45480.0
0,2002-09-03 13:53:00,86.19,27000.0
0,2002-09-04 05:51:00,93.84,57480.0
0,2002-09-04 06:09:00,123.5,1080.0
0,2002-09-05 06:14:00,61.72,86700.0
0,2002-09-05 09:35:00,57.1,12060.0


:::{tip} `EXTRACT(EPOCH FROM interval)` returns seconds
`Datetime - LAG(Datetime)` is an `INTERVAL`. Wrapping it in
`EXTRACT(EPOCH FROM ...)` converts it to a numeric number of seconds, which
is what most downstream models expect. Because `Datetime` has minute
resolution, every gap here is a multiple of 60 seconds.
:::

### A Composite Suspicious-Transaction Flag

This is the SQL version of the fraud rule from the Polars notebook. A transaction is flagged when it is online, its amount is above the user's average amount **so far** (over earlier transactions only), and it happened in the same minute as the user's previous transaction.

The past-only average uses a window frame: `ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING` covers every earlier row of the user and stops just before the current one. For a user's first transaction the frame is empty, so the average is `NULL`, and `COALESCE(..., FALSE)` makes sure the rule does not fire.

We store the query as a view named `flagged`, so the next cells can reuse it.

In [20]:
con.execute(
    """
CREATE OR REPLACE VIEW flagged AS
WITH features AS (
    SELECT
        file_row_number,
        "User",
        "Amount",
        "Use Chip" = 'Online Transaction' AS is_online,
        "Is Fraud?" = 'Yes'               AS is_fraud,
        EXTRACT(EPOCH FROM
            "Datetime" - LAG("Datetime") OVER (
                PARTITION BY "User"
                ORDER BY "Datetime", "Card", file_row_number
            )
        ) AS seconds_since_last_txn,
        AVG("Amount") OVER (
            PARTITION BY "User"
            ORDER BY "Datetime", "Card", file_row_number
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS user_prior_avg_amount
    FROM transactions_numbered
)
SELECT
    *,
    COALESCE(
        is_online
        AND "Amount" > user_prior_avg_amount
        AND seconds_since_last_txn = 0,
        FALSE
    ) AS is_suspicious
FROM features;
"""
)

con.sql(
    """
SELECT
    is_suspicious,
    COUNT(*)                AS n_transactions,
    ROUND(AVG("Amount"), 2) AS avg_amount
FROM flagged
GROUP BY is_suspicious
ORDER BY is_suspicious
"""
)

is_suspiciousBOOLEAN,n_transactionsBIGINT,avg_amountDOUBLE
False,24383426,43.62
True,3474,117.25


### How good is the rule?

As in the Polars notebook, we compare the rule with the `"Is Fraud?"` label. `COUNT(*) FILTER (WHERE ...)` counts only the rows that meet a condition, and DuckDB lets later columns in the `SELECT` refer to earlier ones by name.

In [21]:
con.sql(
    """
SELECT
    COUNT(*) FILTER (WHERE is_suspicious)              AS n_flagged,
    COUNT(*) FILTER (WHERE is_suspicious AND is_fraud) AS n_flagged_fraud,
    COUNT(*) FILTER (WHERE is_fraud)                   AS n_fraud,
    ROUND(100 * n_flagged_fraud / n_flagged, 1)        AS precision_pct,
    ROUND(100 * n_flagged_fraud / n_fraud, 2)          AS recall_pct,
    ROUND(100 * AVG(is_fraud::INTEGER), 2)             AS fraud_rate_pct
FROM flagged
"""
)

n_flaggedBIGINT,n_flagged_fraudBIGINT,n_fraudBIGINT,precision_pctDOUBLE,recall_pctDOUBLE,fraud_rate_pctDOUBLE
3474,119,29757,3.4,0.4,0.12


### Check the result against Polars

The same logic in two engines should give the same answer, so it is worth checking. The cell below runs the Polars version of the rule on the same Parquet file. `with_row_index` numbers the rows in file order, exactly like `file_row_number`. We then compare the two sets of flagged rows. `assert` stops the notebook with an error if they differ.

In [22]:
import polars as pl

polars_flagged = (
    pl.scan_parquet("credit_card_transactions-ibm_v2.parquet")
    .with_row_index("row_nr")
    .sort(["User", "Datetime", "Card", "row_nr"])
    .with_columns(
        (pl.col("Datetime") - pl.col("Datetime").shift(1).over("User"))
        .dt.total_seconds()
        .alias("seconds_since_last_txn"),
        (
            pl.col("Amount").cum_sum().shift(1).over("User")
            / pl.int_range(pl.len()).over("User")
        ).alias("user_prior_avg_amount"),
    )
    .filter(
        (pl.col("Use Chip") == "Online Transaction")
        & (pl.col("Amount") > pl.col("user_prior_avg_amount"))
        & (pl.col("seconds_since_last_txn") == 0)
    )
    .select("row_nr")
    .collect()
)

duckdb_flagged = con.sql(
    "SELECT file_row_number AS row_nr FROM flagged WHERE is_suspicious"
).pl()

polars_rows = set(polars_flagged["row_nr"].to_list())
duckdb_rows = set(duckdb_flagged["row_nr"].to_list())

print(f"Polars flags {len(polars_rows):,} rows.")
print(f"DuckDB flags {len(duckdb_rows):,} rows.")
assert polars_rows == duckdb_rows, "Polars and DuckDB flag different rows"
print("Both engines flag exactly the same rows.")

Polars flags 3,474 rows.
DuckDB flags 3,474 rows.
Both engines flag exactly the same rows.


:::{seealso} Polars and DuckDB express the same idea
Compare the SQL above to the Polars version in the previous notebook. The
business logic is identical; only the syntax changes, and the check above
confirms that both engines flag the same rows. That only holds because both
sort ties the same way. Without the tie-breaker, the two notebooks reported
different counts for the same rule. Picking the right tool is mostly a matter
of taste, team conventions, and whether the data is already loaded as a
DataFrame or sitting in Parquet on disk.
:::

## Summary

In this notebook you repeated the Polars workflow in SQL:

- Queried the 24.4 million-row Parquet file through a view, without loading it into a DataFrame first.
- Counted missing values with `COUNT(*) - COUNT(column)` and found the same three columns as in Polars, with the same business reasons.
- Converted the `'Yes'`/`'No'` fraud label to a boolean and used it to compute fraud rates by channel. Online transactions have a fraud rate of about 0.68%, compared with about 0.08% for chip and 0.04% for swipe.
- Built window features with `PARTITION BY`, `LAG()`, and window frames, using `file_row_number` to break ties in the same minute.
- Rebuilt the composite fraud rule with a past-only average. It flags about 3,500 transactions with a precision of about 3% and a recall well under 1%, and Polars flags exactly the same rows.